In [3]:
#! pip install prometheus-api-client pandas matplotlib

In [ ]:
import pandas as pd
from prometheus_api_client import PrometheusConnect
import matplotlib.pyplot as plt

# 1. Connect to Prometheus
# Use 'http://localhost:9090' if running locally
# Use 'http://prometheus:9090' if inside the same Docker network
prom = PrometheusConnect(url="http://localhost:9090", disable_ssl=True)


In [226]:
# Define our targets and their business explanations
metric_configs = [
    {"q": 0.50, "name": "P50 (Median)", "desc": "The typical latency. 50% of messages are processed faster than this."},
    {"q": 0.95, "name": "P95 (Tail)", "desc": "The 'slow' case. Standard for measuring system performance and stability."},
    {"q": 0.99, "name": "P99 (Extreme)", "desc": "The worst-case scenario. Identifies rare but severe delays or bottlenecks."},
]

table_data = []

for item in metric_configs:
    query = f'histogram_quantile({item["q"]}, sum(rate(message_processing_latency_seconds_bucket[5m])) by (le))'
    result = prom.custom_query(query=query)
    
    if result:
        # We take the first result (aggregated across partitions)
        value = float(result[0]['value'][1])
        table_data.append({
            "Metric": item["name"],
            "Value (Seconds)": f"{value:.4f}",
            "Explanation": item["desc"]
        })

# 3. Create and display the DataFrame
summary_df = pd.DataFrame(table_data)

print("--- Latency Performance Summary ---")
display(summary_df)

--- Latency Performance Summary ---


,Metric,Value (Seconds),Explanation
0,P50 (Median),4.7892,The typical latency. 50% of messages are proce...
1,P95 (Tail),54.4714,The 'slow' case. Standard for measuring system...
2,P99 (Extreme),90.8943,The worst-case scenario. Identifies rare but s...


In [240]:
import pandas as pd
from prometheus_api_client import PrometheusConnect

prom = PrometheusConnect(url="http://localhost:9090", disable_ssl=True)

# Config
metrics = {
    "Total End-to-End": "message_processing_latency_seconds_bucket",
    "Consumer Queue Time": "message_queue_time_seconds_bucket"
}
quantiles = [0.50, 0.95, 0.99]
results = []

for name, metric in metrics.items():
    row = {"Stage": name}
    for q in quantiles:
        query = f'histogram_quantile({q}, sum(rate({metric}[5m])) by (le))'
        res = prom.custom_query(query)
        val = float(res[0]['value'][1]) if res else 0
        row[f"P{int(q*100)}"] = round(val, 4)
    results.append(row)

df = pd.DataFrame(results).set_index("Stage")

# Calculate Internal
internal = (df.loc["Total End-to-End"] - df.loc["Consumer Queue Time"]).clip(lower=0)
internal.name = "Producer Internal"

# Combine and reorder
final_df = pd.concat([df, pd.DataFrame([internal])])
# This line ensures 'Total' is always at the bottom
final_df = final_df.reindex(["Producer Internal", "Consumer Queue Time", "Total End-to-End"]).reset_index()

print("Latency Summary (Seconds):")
display(final_df)

Latency Summary (Seconds):


,index,P50,P95,P99
0,Producer Internal,0.8133,0.6547,0.1310
1,Consumer Queue Time,4.1034,15.4167,19.0833
2,Total End-to-End,4.9167,16.0714,19.2143


In [246]:
import plotly.express as px
from datetime import datetime, timedelta
minutes_interval = 10.0
time_bucket_for_average = 5 #minutes

# 2. Setup Time Range (Last 12 hours)
end_time = datetime.now()
start_time = end_time - timedelta(hours=12)
step = "1m"

query = f'histogram_quantile(0.95, sum(rate(message_processing_latency_seconds_bucket[{time_bucket_for_average}m])) by (le, partition))'

# 3. Fetch Data
result = prom.custom_query_range(
    query=query,
    start_time=start_time,
    end_time=end_time,
    step=step,
)

# 4. Flatten the result into a DataFrame
df_list = []
for series in result:
    partition = series['metric'].get('partition', 'all')
    for val in series['values']:
        df_list.append({
            'timestamp': pd.to_datetime(val[0], unit='s'),
            'partition': f"Partition {partition}",
            'latency_sec': float(val[1])
        })

df = pd.DataFrame(df_list)
df['timestamp'] = pd.to_datetime(df['timestamp']) 

# 5. Create Plotly Graph
fig = px.line(
    df, 
    x='timestamp', 
    y='latency_sec', 
    color='partition',
    title='P95 End-to-End Latency (Interactive)',
    labels={'timestamp': 'Time', 'latency_sec': 'Latency (Seconds)'},
    template='plotly_white'
)

# Convert to milliseconds for Plotly
ms_interval = minutes_interval * 60 * 1000
# Logic to determine the best format
if minutes_interval < 60:
    # Under 1 hour: Show minutes and seconds
    custom_format = "%H:%M:%S"
elif minutes_interval < 1440: # 24 hours
    # 1 to 24 hours: Show Hour:Minute and Short Date
    custom_format = "%H:%M\n%b %d"
else:
    # Over 24 hours: Focus on the Date
    custom_format = "%b %d\n%Y"

fig.update_xaxes(
    type='date',             # Force the axis to recognize dates
    tickmode='linear',
    tick0=df['timestamp'].min(), 
    dtick=ms_interval,
    tickformat=custom_format,
    ticks="outside",
    ticklen=10,
    tickcolor='black',
    showgrid=True,
    gridcolor='LightPink'
)
fig.show()

# 6. Display Stats Table
print("--- Performance Stats ---")
stats_table = df.groupby('partition')['latency_sec'].agg(['mean', 'max', 'std']).reset_index()
display(stats_table)

--- Performance Stats ---


,partition,mean,max,std
0,Partition all,103.3393,178.333333,55.389542


In [ ]:
import pandas as pd
from prometheus_api_client import PrometheusConnect
import matplotlib.pyplot as plt

# 1. Connect to Prometheus
# Use 'http://localhost:9090' if running locally
# Use 'http://prometheus:9090' if inside the same Docker network
prom = PrometheusConnect(url="http://localhost:9090", disable_ssl=True)

# 2. Define the Query (P95 Latency)
query = 'histogram_quantile(0.95, sum(rate(message_processing_latency_seconds_bucket[5m])) by (le))'

# 3. Fetch Data as a list of dicts and convert to DataFrame
result = prom.custom_query(query=query)
df_list = []

for series in result:
    partition = series['metric'].get('partition', 'all')
    value = float(series['value'][1])
    timestamp = pd.to_datetime(series['value'][0], unit='s')
    df_list.append({'timestamp': timestamp, 'partition': partition, 'latency_sec': value})

df = pd.DataFrame(df_list)

# 4. Display the Table of Stats
print("--- Latency Statistics per Partition ---")
stats_table = df.groupby('partition')['latency_sec'].agg(['mean', 'max', 'min', 'std']).reset_index()
display(stats_table)

# 5. Simple Plot
df.pivot(index='timestamp', columns='partition', values='latency_sec').plot(kind='line', figsize=(10, 5))
plt.title("P95 Latency over Time")
plt.ylabel("Seconds")
plt.show()